In [1]:
# Use OpenAI GPT-4.1 via LangGraph to extract the core conversation content from each message in df['content']
# and summarize it in a sentence.

from openai import OpenAI
import os
from dotenv import load_dotenv
import json
import os
import sys
import numpy as np
import uuid
import pandas as pd
from py2neo import Graph
import textwrap

# Load environment variables from secret/.env
load_dotenv(dotenv_path="secret/.env")

api_key = os.getenv("OPENAI_API_KEY")
# Set your OpenAI API key (ensure you have it in your environment or replace with your key)
client = OpenAI(api_key=api_key)
from langchain_openai import ChatOpenAI,OpenAIEmbeddings

In [2]:
embeddings_df=pd.read_csv("data/message_embeddings.csv")

In [3]:
topic_model_df=pd.read_csv("data/topic_model.csv")

In [4]:
tmodel_df=topic_model_df.loc[:,['source','target','source_entity_type','target_entity_type']].drop_duplicates()

In [21]:
# Create a DataFrame of all unique entities and their types from the 4 columns

# Stack the source and target columns with their corresponding entity types
entities = pd.concat([
    tmodel_df[['source', 'source_entity_type']].rename(columns={'source': 'entity', 'source_entity_type': 'entity_type'}),
    tmodel_df[['target', 'target_entity_type']].rename(columns={'target': 'entity', 'target_entity_type': 'entity_type'})
])

# Drop duplicates to get unique entities and their types
unique_entities_df = entities.drop_duplicates().reset_index(drop=True)
unique_entities_df

,entity,entity_type
0,the lookout,Person
1,the intern,Person
2,kelly,Person
3,mrs. money,Person
4,boss,Person
5,the middleman,Person
6,serenity,Vessel
7,mako,Vessel
8,himark harbor,Location
9,reef guardian,Vessel


In [27]:
unique_entities_df.to_csv("data/unique_entities_df.csv",index=False)

In [4]:
topics=list(set(topic_model_df['topic']))
topics

['Boundary Violations Monitoring',
 'Reef Surveillance',
 'Illegal Resource Extraction',
 'Project Coordination Schedule',
 'Birdwatching',
 'Security & Docking',
 'Music Video Shooting',
 'Vessel Operations Coordination',
 'Corruption Investigation',
 'Vessel Identification Compliance']

In [5]:
# topic='Illegal Resource Extraction'

In [6]:
# tdf=topic_model_df.loc[topic_model_df['topic']==topic]


In [7]:
# # Find rows in tdf where consecutive Message_IDs have at least 3 unique factors among [source, target] of both messages

# # Ensure Message_ID is sorted and reset index for consecutive comparison
# tdf_sorted = topic_model_df.sort_values("Message_ID").reset_index(drop=True)

# matching_indices = []
# for i in range(len(tdf_sorted) - 1):
#     row1 = tdf_sorted.iloc[i]
#     row2 = tdf_sorted.iloc[i + 1]
#     # Check if Message_IDs are consecutive
#     if row2["Message_ID"] == row1["Message_ID"] + 1:
#         # Collect source and target from both rows
#         factors = [row1["source"], row1["target"], row2["source"], row2["target"]]
#         # Count unique factors
#         if len(set(factors)) >= 3:
#             matching_indices.extend([i, i+1])

# # Get unique indices and select rows
# matching_indices = sorted(set(matching_indices))
# tdf_matches = tdf_sorted.iloc[matching_indices]
# tdf_matches

In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Assume embeddings_df is already loaded and has columns: 'Message_ID', 'embedding'
# Convert 'embedding' column from string to list if needed
if isinstance(embeddings_df['embedding'].iloc[0], str):
    embeddings_df['embedding'] = embeddings_df['embedding'].apply(eval)

# Prepare a list to store results
cosine_sim_results = []

# For all pairs where source_id < target_id
for i, row_i in embeddings_df.iterrows():
    source_id = row_i['Message_ID']
    source_emb = np.array(row_i['embedding']).reshape(1, -1)
    for j, row_j in embeddings_df.iterrows():
        target_id = row_j['Message_ID']
        if source_id < target_id:
            target_emb = np.array(row_j['embedding']).reshape(1, -1)
            sim = cosine_similarity(source_emb, target_emb)[0][0]
            cosine_sim_results.append({
                'source_id': source_id,
                'target_id': target_id,
                'cosine_similarity': sim
            })

# Convert results to DataFrame
cosine_sim_df = pd.DataFrame(cosine_sim_results)
cosine_sim_df.head()

,source_id,target_id,cosine_similarity
0,1,2,0.681825
1,1,3,0.610072
2,1,4,0.470849
3,1,5,0.211254
4,1,6,0.279851


In [25]:
# Function to get cosine similarity for a given source_id and target_id pair
def get_cosine_similarity(source_id, target_id):
    result = cosine_sim_df[
        (cosine_sim_df['source_id'] == source_id) & 
        (cosine_sim_df['target_id'] == target_id)
    ]
    if not result.empty:
        return result['cosine_similarity'].iloc[0]
    else:
        return None

# Example usage:
source_id = 492
target_id = 493
sim = get_cosine_similarity(source_id, target_id)
print(f"Cosine similarity between {source_id} and {target_id}: {sim}")

Cosine similarity between 492 and 493: 0.644025203398241


In [9]:
# Set the cosine similarity threshold
threshold = 0.7

# Find all rows where source_id and target_id are consecutive and similarity >= threshold
consecutive_sim_df = cosine_sim_df[
    (cosine_sim_df['target_id'] == cosine_sim_df['source_id'] + 1) &
    (cosine_sim_df['cosine_similarity'] >= threshold)
]

consecutive_sim_df

,source_id,target_id,cosine_similarity
2310,5,6,0.768837
2885,6,7,0.714696
4032,8,9,0.788990
4604,9,10,0.732851
5175,10,11,0.804287
...,...,...,...
167679,559,560,0.836418
167790,565,566,0.772637
167844,569,570,0.743676
167855,570,571,0.704133


In [10]:
# # Check source and target mappings for consecutive_sim_df

# consecutive_sim_df = consecutive_sim_df.copy()

# Use tdf_matches for mapping, fallback to topic_model_df if needed
# mapping_df = topic_model_df if 'Message_ID' in topic_model_df.columns else topic_model_df

# Map processed text
consecutive_sim_df['source_message'] = consecutive_sim_df['source_id'].map(
    topic_model_df.set_index('Message_ID')['Processed_Text']
)
consecutive_sim_df['target_message'] = consecutive_sim_df['target_id'].map(
    topic_model_df.set_index('Message_ID')['Processed_Text']
)

# # Map packet IDs
# consecutive_sim_df['source_packet_id'] = consecutive_sim_df['source_id'].map(
#     mapping_df.set_index('Message_ID')['packet_id']
# )
# consecutive_sim_df['target_packet_id'] = consecutive_sim_df['target_id'].map(
#     mapping_df.set_index('Message_ID')['packet_id']
# )

# # Map source and target columns
# consecutive_sim_df['source'] = consecutive_sim_df['source_id'].map(
#     mapping_df.set_index('Message_ID')['source']
# )
# consecutive_sim_df['target'] = consecutive_sim_df['target_id'].map(
#     mapping_df.set_index('Message_ID')['target']
# )

# # Check a few rows to verify mappings
# print(consecutive_sim_df[['source_id', 'target_id', 'source', 'target', 'source_message', 'target_message']].head())

consecutive_sim_df

/var/folders/zn/kgdwhjsj6fx6m4zbd82g2p1m0000gn/T/ipykernel_71997/3741232585.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  consecutive_sim_df['source_message'] = consecutive_sim_df['source_id'].map(
/var/folders/zn/kgdwhjsj6fx6m4zbd82g2p1m0000gn/T/ipykernel_71997/3741232585.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  consecutive_sim_df['target_message'] = consecutive_sim_df['target_id'].map(


,source_id,target_id,cosine_similarity,source_message,target_message
2310,5,6,0.768837,i've reviewed our operational funding for the ...,i'm available tomorrow at 3 pm to discuss fund...
2885,6,7,0.714696,i'm available tomorrow at 3 pm to discuss fund...,i'll bring the updated projections for tomorro...
4032,8,9,0.788990,i'd like to move our meeting forward to discus...,i can meet earlier as requested. nemo reef dis...
4604,9,10,0.732851,i can meet earlier as requested. nemo reef dis...,let's meet tomorrow at 4pm at the usual spot b...
5175,10,11,0.804287,let's meet tomorrow at 4pm at the usual spot b...,i'll be at the marina tomorrow at 4pm. just re...
...,...,...,...,...,...
167679,559,560,0.836418,operation complete at nemo reef. all equipment...,well done on nemo reef operation completion. g...
167790,565,566,0.772637,drone footage confirms video production at nem...,can confirm your drone footage findings - defi...
167844,569,570,0.743676,need you at south dock by 2200 to supervise eq...,i'll be at south dock at 2200 as requested for...
167855,570,571,0.704133,i'll be at south dock at 2200 as requested for...,will arrive at south dock with team at 2100 ho...


In [11]:
# Create Source_ID_source and Source_ID_target columns
consecutive_sim_df['Source_ID_source'] = consecutive_sim_df['source_id'].map(
    topic_model_df.set_index('Message_ID')['source']
)
consecutive_sim_df['Source_ID_target'] = consecutive_sim_df['source_id'].map(
    topic_model_df.set_index('Message_ID')['target']
)

# Create Target_ID_source and Target_ID_target columns
consecutive_sim_df['Target_ID_source'] = consecutive_sim_df['target_id'].map(
    topic_model_df.set_index('Message_ID')['source']
)
consecutive_sim_df['Target_ID_target'] = consecutive_sim_df['target_id'].map(
    topic_model_df.set_index('Message_ID')['target']
)

consecutive_sim_df

/var/folders/zn/kgdwhjsj6fx6m4zbd82g2p1m0000gn/T/ipykernel_71997/98047826.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  consecutive_sim_df['Source_ID_source'] = consecutive_sim_df['source_id'].map(
/var/folders/zn/kgdwhjsj6fx6m4zbd82g2p1m0000gn/T/ipykernel_71997/98047826.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  consecutive_sim_df['Source_ID_target'] = consecutive_sim_df['source_id'].map(
/var/folders/zn/kgdwhjsj6fx6m4zbd82g2p1m0000gn/T/ipykernel_71997/98047826.py:10: SettingWithCopyWarning:

,source_id,target_id,cosine_similarity,source_message,target_message,Source_ID_source,Source_ID_target,Target_ID_source,Target_ID_target
2310,5,6,0.768837,i've reviewed our operational funding for the ...,i'm available tomorrow at 3 pm to discuss fund...,mrs. money,boss,boss,mrs. money
2885,6,7,0.714696,i'm available tomorrow at 3 pm to discuss fund...,i'll bring the updated projections for tomorro...,boss,mrs. money,mrs. money,boss
4032,8,9,0.788990,i'd like to move our meeting forward to discus...,i can meet earlier as requested. nemo reef dis...,boss,the middleman,the middleman,boss
4604,9,10,0.732851,i can meet earlier as requested. nemo reef dis...,let's meet tomorrow at 4pm at the usual spot b...,the middleman,boss,boss,the middleman
5175,10,11,0.804287,let's meet tomorrow at 4pm at the usual spot b...,i'll be at the marina tomorrow at 4pm. just re...,boss,the middleman,the middleman,boss
...,...,...,...,...,...,...,...,...,...
167679,559,560,0.836418,operation complete at nemo reef. all equipment...,well done on nemo reef operation completion. g...,mako,davis,davis,mako
167790,565,566,0.772637,drone footage confirms video production at nem...,can confirm your drone footage findings - defi...,ecovigil,sentinel,sentinel,ecovigil
167844,569,570,0.743676,need you at south dock by 2200 to supervise eq...,i'll be at south dock at 2200 as requested for...,davis,rodriguez,small fry,davis
167855,570,571,0.704133,i'll be at south dock at 2200 as requested for...,will arrive at south dock with team at 2100 ho...,small fry,davis,davis,remora


In [12]:
# # Identify rows where at least one of Target_ID_source or Target_ID_target does not appear in Source_ID_source or Source_ID_target
# mask = ~(
#     consecutive_sim_df['Target_ID_source'].isin(consecutive_sim_df['Source_ID_source']) &
#     consecutive_sim_df['Target_ID_target'].isin(consecutive_sim_df['Source_ID_target'])
# )
# rows_with_unique_targets = consecutive_sim_df[mask]
# rows_with_unique_targets

In [14]:
consecutive_sim_df.to_csv("consecutive_sim_df.csv",index=False)

In [5]:
consecutive_sim_df=pd.read_csv("consecutive_sim_df.csv")

In [7]:
# Remove rows where Source_ID_source == Target_ID_target AND Source_ID_target == Target_ID_source
filtered_sim_df = consecutive_sim_df[
    ~(
        (consecutive_sim_df['Source_ID_source'] == consecutive_sim_df['Target_ID_target']) &
        (consecutive_sim_df['Source_ID_target'] == consecutive_sim_df['Target_ID_source'])
    )
]
filtered_sim_df

,source_id,target_id,cosine_similarity,source_message,target_message,Source_ID_source,Source_ID_target,Target_ID_source,Target_ID_target
5,17,18,0.743267,"after speaking with the middleman, i'm confide...",i've prepared those financial projections we d...,boss,mrs. money,mrs. money,the middleman
7,24,25,0.847575,reporting increased tourism vessel activity ne...,we've received reports from paackland harbor a...,paackland harbor,oceanus city council,oceanus city council,liam thorne
9,38,39,0.744049,confirming nemo reef as primary location. need...,submitting a permit application for a music vi...,glitters team,samantha blake,samantha blake,oceanus city council
10,40,41,0.797579,while patrolling western boundary of protected...,we're seeing increased tourism activity near p...,sentinel,green guardians,green guardians,horizon
11,41,42,0.760762,we're seeing increased tourism activity near p...,received coordinates from green guardians abou...,green guardians,horizon,horizon,sentinel
...,...,...,...,...,...,...,...,...,...
148,533,534,0.711981,following your instruction to maintain distanc...,unauthorized vessels detected at nemo reef ope...,sentinel,mako,mako,davis
153,547,548,0.735736,vessel prepared for your crew's arrival at 060...,confirming equipment needs for tomorrow: bring...,remora,sailor shifts team,glitters team,remora
156,569,570,0.743676,need you at south dock by 2200 to supervise eq...,i'll be at south dock at 2200 as requested for...,davis,rodriguez,small fry,davis
157,570,571,0.704133,i'll be at south dock at 2200 as requested for...,will arrive at south dock with team at 2100 ho...,small fry,davis,davis,remora


In [9]:
filtered_sim_df.to_csv("data/pseduonyms_finder.csv",index=False)